# 03 — Reviewer validation sampling

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 03 — Blinded semantic-pair validation sampling and post-length audit

Run after Notebook 01. This public version contains only the reviewer-validation components used in the final revised manuscript.


## 0 — Configuration

In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

UPSTREAM_RUN_NAME = "main_pipeline_no_brain_fog_final_v1"
VALIDATION_RUN_NAME = "reviewer_validation_no_brain_fog_final_v1"

UPSTREAM_DIR = str(PROJECT_ROOT / "outputs" / UPSTREAM_RUN_NAME)
SAVE_DIR = str(PROJECT_ROOT / "outputs" / VALIDATION_RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

PAIR_SAMPLE_PER_STRATUM = 50
PAIR_RANDOM_SEED = 20260907
PAIR_BLOCK_SIZE = 512

print("Frozen upstream folder :", UPSTREAM_DIR)
print("Validation output      :", SAVE_DIR)


## 1 — Load frozen data and reconstruct embedding order

In [ ]:

prop_path = os.path.join(
    UPSTREAM_DIR,
    "filtered_posts_df_2_corrected_propagated.parquet"
)
emb_path = os.path.join(
    UPSTREAM_DIR,
    "embeddings_revision.npy"
)
sem_map_path = os.path.join(
    UPSTREAM_DIR,
    "semantic_node_topic_map_revision.csv"
)

for p in [prop_path, emb_path, sem_map_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            "Missing Notebook 01 output: " + p
        )

fp = pd.read_parquet(prop_path)
embeddings = normalize(
    np.load(emb_path),
    norm="l2"
).astype(np.float32)
semantic_node_topic_map = pd.read_csv(sem_map_path)

# Reconstruct exactly the deduplicated text order used in Notebook 01:
# non-empty processed text, then drop_duplicates preserving first occurrence.
nonempty = fp.loc[
    fp["text_processed"].notna()
    & fp["text_processed"].astype(str).str.strip().ne("")
].copy()

dedup = nonempty.drop_duplicates(
    subset=["text_processed"],
    keep="first"
).copy()

dedup = dedup.reset_index(drop=True)

if len(dedup) != embeddings.shape[0]:
    raise RuntimeError(
        f"Embedding/text order mismatch: {embeddings.shape[0]} embeddings "
        f"but {len(dedup)} reconstructed unique processed texts."
    )

# Strong cross-check: semantic_node_id must point to the same original_index.
check_map = semantic_node_topic_map.copy()
check_map["semantic_node_id"] = check_map["semantic_node_id"].astype(int)

mapped_orig = dedup.loc[
    check_map["semantic_node_id"].to_numpy(),
    "original_index"
].to_numpy()

if not np.array_equal(
    mapped_orig,
    check_map["original_index"].to_numpy()
):
    raise RuntimeError(
        "Reconstructed deduplicated text order does not match Notebook 01 semantic mapping."
    )

print(f"Unique processed texts / embeddings: {len(dedup):,}")
print("✅ Embedding order verified against semantic_node_topic_map.")


## 2 — Reviewer 1.6: prepare a blinded semantic-pair validation sample

Four prespecified cosine-similarity strata are used:

- 0.60–<0.65
- 0.65–<0.70
- 0.70–<0.75
- 0.75–0.85

The code scans the frozen embedding similarities once and uses random-key reservoir
sampling so each eligible pair has an equal chance to remain in its stratum sample.

The **BLINDED** sheet contains no cosine score or stratum.


In [ ]:

PAIR_STRATA = [
    ("0.60_to_lt_0.65", 0.60, 0.65, False),
    ("0.65_to_lt_0.70", 0.65, 0.70, False),
    ("0.70_to_lt_0.75", 0.70, 0.75, False),
    ("0.75_to_0.85",    0.75, 0.85, True),
]

rng = np.random.default_rng(PAIR_RANDOM_SEED)

# Each reservoir stores rows with the smallest random keys seen so far.
reservoirs = {
    name: pd.DataFrame(
        columns=["random_key", "i", "j", "cosine_similarity"]
    )
    for name, _, _, _ in PAIR_STRATA
}

n_docs = embeddings.shape[0]

for start in range(0, n_docs, PAIR_BLOCK_SIZE):
    stop = min(start + PAIR_BLOCK_SIZE, n_docs)
    sims_block = embeddings[start:stop] @ embeddings.T

    for local_i in range(stop - start):
        i = start + local_i
        if i + 1 >= n_docs:
            continue

        js = np.arange(i + 1, n_docs)
        vals = sims_block[local_i, i + 1:]

        for name, low, high, include_high in PAIR_STRATA:
            if include_high:
                mask = (vals >= low) & (vals <= high)
            else:
                mask = (vals >= low) & (vals < high)

            if not np.any(mask):
                continue

            cand_j = js[mask]
            cand_sim = vals[mask]

            candidate_df = pd.DataFrame({
                "random_key": rng.random(len(cand_j)),
                "i": np.full(len(cand_j), i, dtype=int),
                "j": cand_j.astype(int),
                "cosine_similarity": cand_sim.astype(float),
            })

            combined = pd.concat(
                [reservoirs[name], candidate_df],
                ignore_index=True
            )
            reservoirs[name] = (
                combined
                .nsmallest(PAIR_SAMPLE_PER_STRATUM, "random_key")
                .reset_index(drop=True)
            )

    print(f"Scanned {stop:,}/{n_docs:,} unique texts", end="\r")

print()

sample_parts = []
for name, low, high, include_high in PAIR_STRATA:
    part = reservoirs[name].copy()
    if len(part) < PAIR_SAMPLE_PER_STRATUM:
        print(
            f"WARNING: stratum {name} yielded only {len(part)} sampled pairs "
            f"(target={PAIR_SAMPLE_PER_STRATUM})."
        )
    part["similarity_stratum"] = name
    sample_parts.append(part)

pair_key = pd.concat(sample_parts, ignore_index=True)

pair_key["text_a"] = pair_key["i"].map(
    dedup["text"].astype(str).to_dict()
)
pair_key["text_b"] = pair_key["j"].map(
    dedup["text"].astype(str).to_dict()
)

# Randomize review order across strata.
pair_key = (
    pair_key
    .sample(frac=1.0, random_state=PAIR_RANDOM_SEED)
    .reset_index(drop=True)
)
pair_key.insert(0, "pair_id", np.arange(1, len(pair_key) + 1))

answer_key_cols = [
    "pair_id", "i", "j", "cosine_similarity",
    "similarity_stratum", "text_a", "text_b"
]
pair_key[answer_key_cols].to_csv(
    os.path.join(SAVE_DIR, "threshold_pair_validation_ANSWER_KEY.csv"),
    index=False
)

blinded = pair_key[["pair_id", "text_a", "text_b"]].copy()
blinded["reviewer_1_similar"] = ""
blinded["reviewer_2_similar"] = ""
blinded["reviewer_1_notes"] = ""
blinded["reviewer_2_notes"] = ""

blinded_path = os.path.join(
    SAVE_DIR,
    "threshold_pair_validation_BLINDED.csv"
)
blinded.to_csv(blinded_path, index=False)

print("Pair validation sample created:", len(blinded))
print("BLINDED review file :", blinded_path)
print(
    "Use 1 = meaningfully semantically similar; "
    "0 = not meaningfully semantically similar."
)


## 4 — Reviewer 2.6: post-length audit

In [ ]:

post_length = (
    fp[["post_id", "text"]]
    .dropna(subset=["text"])
    .drop_duplicates(subset=["post_id"])
    .copy()
)

post_length["character_count"] = post_length["text"].astype(str).str.len()
post_length["word_count"] = (
    post_length["text"]
    .astype(str)
    .str.split()
    .str.len()
)

def summarize_length(series, metric):
    return {
        "metric": metric,
        "n": int(series.notna().sum()),
        "mean": float(series.mean()),
        "sd": float(series.std()),
        "min": float(series.min()),
        "p25": float(series.quantile(0.25)),
        "median": float(series.median()),
        "p75": float(series.quantile(0.75)),
        "p95": float(series.quantile(0.95)),
        "max": float(series.max()),
    }

post_length_summary = pd.DataFrame([
    summarize_length(post_length["character_count"], "characters"),
    summarize_length(post_length["word_count"], "words"),
])

display(post_length_summary)

post_length_summary.to_csv(
    os.path.join(SAVE_DIR, "post_length_summary.csv"),
    index=False
)

post_length.to_csv(
    os.path.join(SAVE_DIR, "post_length_post_level.csv"),
    index=False
)

print(
    "Posts with character_count >= 290:",
    int((post_length["character_count"] >= 290).sum())
)
print(
    "Posts with character_count >= 300:",
    int((post_length["character_count"] >= 300).sum())
)


## Completion checklist


In [ ]:
print("03 preparation run completed.")
print("Expected outputs in validation folder:")
for f in [
    "threshold_pair_validation_BLINDED.csv",
    "threshold_pair_validation_ANSWER_KEY.csv",
    "post_length_summary.csv",
    "post_length_post_level.csv",
]:
    print(" -", f)
